#### Extraction

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import polars as pl

 ##### Dataframe statique

In [ ]:

path = "../Datasets/clean_full_static_ano.parquet"
df_static = pl.read_parquet(path)
df_static = df_static.with_columns(pl.col("encounterId").cast(pl.Int32))

##### Dataframe dynamique

In [ ]:
from Extraction import extract
path = "../Datasets/temporal_tailored_imputation.parquet"
df_test = extract.extract_data_survie(path)

#### Transformation_Prétraitement

Il faudra changer hour_offset pour pouvoir prendre une date fixe et non juste un temps en arrière

ajout de la colonne age qui est dans le thesaurus

In [ ]:
df_test = df_test.join(
    df_static[["encounterId", "age"]],
    on="encounterId",
    how="left"
)

In [ ]:
from Transformation_Pretraitement import preprocessing_polars
df_clean = preprocessing_polars.prepare_data(df_test,0)

In [ ]:
# from Transformation_Pretraitement import preprocessing_new
# df_test2 = df_test.to_pandas()
# df_clean2 = preprocessing_new.prepare_data(df_test2,0)

In [ ]:
import pandas as pd
 
pd.testing.assert_frame_equal(
    df_clean.to_pandas().sort_index(axis=1),
    df_clean2.sort_index(axis=1),
    check_dtype=False
)

#### Préparation pour InceptionTime

In [ ]:
import numpy as np
import pandas as pd
 
 
def df_to_inceptiontime_tensor(
    df,
    patient_col="encounterId",
    time_col="heure_calibree",
    target_col="non_survival",
    feature_cols=None,
    expected_length=24,
):
    df = df.copy()
 
    # Si les features ne sont pas fournies, on prend toutes les colonnes utiles
    if feature_cols is None:
        excluded = {patient_col, time_col, target_col}
        feature_cols = [c for c in df.columns if c not in excluded]
 
    # On garde seulement les patients ayant exactement expected_length lignes
    counts = df.groupby(patient_col).size()
    valid_ids = counts[counts == expected_length].index
 
    df = df[df[patient_col].isin(valid_ids)].copy()
 
    if df.empty:
        raise ValueError("Aucun patient n'a exactement la longueur attendue.")
 
    # Tri obligatoire
    df = df.sort_values([patient_col, time_col])
 
    X_list = []
    y_list = []
    kept_ids = []
 
    for pid, g in df.groupby(patient_col):
        g = g.sort_values(time_col)
 
        # features : shape (time, features)
        x = g[feature_cols].to_numpy(dtype=np.float32)
 
        # InceptionTime attend en général (features, time)
        x = x.T
 
        # label patient
        y = g[target_col].iloc[0]
 
        X_list.append(x)
        y_list.append(y)
        kept_ids.append(pid)
 
    if not X_list:
        raise ValueError("Impossible de construire le tenseur X.")
 
    X = np.stack(X_list, axis=0)   # (n_samples, n_features, time)
    y = np.asarray(y_list)
 
    return X, y, feature_cols, kept_ids

In [ ]:
df_clean = df_clean.join(
    df_static[["encounterId", "isDeceased"]],
    on="encounterId",
    how="left"
)

In [ ]:
df_inception = df_clean.to_pandas()
X, y, feature_cols, kept_ids = df_to_inceptiontime_tensor(
    df_inception,
    patient_col="encounterId",
    time_col="heure_calibree",
    target_col="isDeceased",
    expected_length=24,
)

##### Découpage en train/test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y # stratify permet de conserver les proportions initiales entre décès et survie
)
 
n_train, n_features, time_len = X_train.shape
n_test = X_test.shape[0]
 
scaler = StandardScaler()
 
X_train_2d = X_train.transpose(0, 2, 1).reshape(-1, n_features)
X_test_2d = X_test.transpose(0, 2, 1).reshape(-1, n_features)
 
X_train_scaled = scaler.fit_transform(X_train_2d)
X_test_scaled = scaler.transform(X_test_2d)
 
X_train = X_train_scaled.reshape(n_train, time_len, n_features).transpose(0, 2, 1)
X_test = X_test_scaled.reshape(n_test, time_len, n_features).transpose(0, 2, 1)

#### Training sur InceptionTime

In [ ]:
from inceptionTimeModified import train_inception_time

model, T, history, splits = train_inception_time(
    X_train, y_train,
    epochs=100,
    patience=10,
    save_best_path="models/inception_test.pt"
)

In [ ]:
from inceptionTimeModified import evaluate_on_test

auc, brier, T = evaluate_on_test(
    X_test, y_test,
    "models/inception_test.pt"
)

from inceptionTimeModified import predict_proba, load_model_from_checkpoint

model, _, T = load_model_from_checkpoint("models/inception_test.pt")

In [ ]:
probas = predict_proba(model, X_test, T=T)
print(probas)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
 
fpr, tpr, thresholds = roc_curve(y_test, probas)
auc = roc_auc_score(y_test, probas)
 
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Hasard")
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs")
plt.title("Courbe ROC")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
 
plt.figure()
 
sns.kdeplot(probas[y_test == 0], label="Survivants", fill=True)
sns.kdeplot(probas[y_test == 1], label="Décès", fill=True)
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Densité")
plt.title("Distribution des scores (KDE)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
 
plt.figure()
 
# Survivants
data_0 = probas[y_test == 0]
sns.kdeplot(data_0, color="lightblue")
x0, y0 = plt.gca().lines[-1].get_data()
y0 = y0 * len(data_0)  # conversion densité → counts
plt.plot(x0, y0, color="blue", label="Survivants")
plt.fill_between(x0, y0, alpha=0.3, color="lightblue")
 
# Décès
data_1 = probas[y_test == 1]
sns.kdeplot(data_1, color="orange")
x1, y1 = plt.gca().lines[-1].get_data()
y1 = y1 * len(data_1)
plt.plot(x1, y1, color="orange", label="Décès")
plt.fill_between(x1, y1, alpha=0.3, color="orange")
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Nombre de patients")
plt.title("Distribution des scores")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.style.use("seaborn-v0_8")
plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
 
prob_true, prob_pred = calibration_curve(y_test, probas, n_bins=10)
 
plt.figure()
plt.plot(prob_pred, prob_true, marker="o", label="Modèle")
plt.plot([0, 1], [0, 1], "--", label="Calibration idéale")
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Fréquence observée")
plt.title("Calibration curve")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
 
thresholds = np.linspace(0.1, 0.9, 50)
f1s = []
 
for t in thresholds:
    y_pred = (probas >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred))
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t
 
import matplotlib.pyplot as plt
 
plt.plot(thresholds, f1s)
plt.xlabel("Threshold")
plt.ylabel("F1 score")
plt.title("F1 vs Threshold")
plt.grid()
plt.show()

print(f"Le meilleur f1 score de{best_f1 : .2f} est atteint lorsque le threshold est égal à{best_t : .2f}")

In [ ]:
threshold = 0.46
y_pred = (probas >= threshold).astype(int)
 
cm = confusion_matrix(y_test, y_pred)
 
plt.figure()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title(f"Confusion matrix (threshold={threshold})")
plt.show()